# Day 6 — Retrieval-augmented drafting (RAG)

**Goal:** given a query image (via fused retrieval), retrieve top-k similar cases and produce a grounded draft impression with citations.

In [ ]:
!pip -q install pandas numpy faiss-cpu

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Load clean dataset + fused index

In [ ]:

import faiss

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
fused = np.load(f"{BASE}/fused_embeddings_alpha_0.5.npy").astype("float32")
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)

index = faiss.IndexFlatIP(fused.shape[1])
index.add(fused)
print("Loaded index size:", index.ntotal)


## 2) Deterministic grounded drafting (no external LLM required)

In [ ]:

def retrieve_cases(query_idx: int, k=3):
    q = fused[query_idx:query_idx+1]
    scores, inds = index.search(q, k+1)
    inds = [i for i in inds[0].tolist() if i != query_idx][:k]
    out = []
    for rank, i in enumerate(inds, start=1):
        out.append({
            "case": rank,
            "row_idx": i,
            "study_id": int(df.loc[i,"study_id"]),
            "score": float(scores[0][rank]),  # note: rank offset
            "impression": df.loc[i,"impression"]
        })
    best = float(scores[0][1]) if len(scores[0]) > 1 else 0.0
    return out, best

def deterministic_impression(cases, max_bullets=2):
    # Simple: take first 1-2 sentences from top cases, add citations.
    bullets = []
    for c in cases:
        s = c["impression"].replace("\n"," ").strip()
        # take first sentence-ish chunk
        first = re.split(r"(?<=[.!?])\s+", s)[0].strip()
        if first:
            bullets.append(f"- {first} [Case {c['case']}]")
        if len(bullets) >= max_bullets:
            break
    if not bullets:
        return "IMPRESSION:\n- Unable to draft from retrieved evidence."
    return "IMPRESSION:\n" + "\n".join(bullets)

query_idx = 0
cases, best = retrieve_cases(query_idx, k=3)
draft = deterministic_impression(cases, max_bullets=2)

print("best_score:", best)
for c in cases:
    print(f"[Case {c['case']}] (score={c['score']:.3f})", c["impression"][:140].replace("\n"," "), "...")
print("\n" + draft)


✅ **Day 6 deliverable:** a working grounded drafting function using retrieved evidence + citations.